In [5]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Load the CSV (replace with your actual file path)
# Assuming it's the combined headline topics file
data = pd.read_csv("../data/topic_modeling/all_sources_headline_topics_sklearn.csv")

# Create a new column indicating if 'trump' is in the topic name (case insensitive)
data['has_trump'] = data['topic'].str.lower().str.contains('coronavirus|covid')

# Filter to only include topics ranked 1-3
top3_data = data[data['rank'] <= 3]

# Group by source and calculate the percentage of top 3 topics that mention Trump
trump_analysis = top3_data.groupby('source').agg(
    total_top3_topics=('topic', 'count'),
    trump_topics=('has_trump', 'sum')
).reset_index()

# Calculate percentage
trump_analysis['trump_percentage'] = (trump_analysis['trump_topics'] / 
                                     trump_analysis['total_top3_topics'] * 100).round(2)

# Add analysis by year to see temporal patterns
yearly_trump = top3_data.groupby(['source', 'year']).agg(
    total_topics=('topic', 'count'),
    trump_topics=('has_trump', 'sum')
).reset_index()

yearly_trump['percentage'] = (yearly_trump['trump_topics'] / 
                             yearly_trump['total_topics'] * 100).round(2)

# Create visualizations
plt.figure(figsize=(10, 6))
sns.barplot(x='source', y='trump_percentage', data=trump_analysis)
plt.title('Percentage of Top 3 Topics Mentioning "Trump" by News Source')
plt.ylabel('Percentage (%)')
plt.xlabel('News Source')
plt.savefig('trump_topic_by_source.png')
plt.close()

# Create a heatmap for yearly analysis
pivot_data = yearly_trump.pivot(index='source', columns='year', values='percentage').fillna(0)
plt.figure(figsize=(14, 6))
sns.heatmap(pivot_data, cmap='YlOrRd', annot=True, fmt='.1f',
            linewidths=.5, cbar_kws={'label': 'Percentage (%)'})
plt.title('Percentage of Top 3 Topics Mentioning "Trump" by Source and Year')
plt.savefig('trump_topic_by_year_source.png')
plt.close()

# Print the overall results
print(trump_analysis)
print("\nPercentage of top 3 topics mentioning 'Trump' by source and year:")
print(pivot_data)

  source  total_top3_topics  trump_topics  trump_percentage
0    abc                369            21              5.69
1    fox                333             9              2.70
2  msnbc                366             6              1.64

Percentage of top 3 topics mentioning 'Trump' by source and year:
year    2015  2016  2017  2018  2019   2020   2021  2022  2023  2024  2025
source                                                                    
abc      0.0   0.0   0.0   0.0   0.0  33.33  16.67  8.33   0.0   0.0   0.0
fox      0.0   0.0   0.0   0.0   0.0   0.00  25.00  0.00   0.0   0.0   0.0
msnbc    0.0   0.0   0.0   0.0   0.0   8.33   8.33  0.00   0.0   0.0   0.0
